# Run the full pipeline (data, baseline, DistilBERT, audit)
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **One-time setup:** click the 🔑 key icon in the left sidebar, add a secret named `GITHUB_TOKEN`, paste a GitHub token that can read the repo, and turn on "Notebook access". (Make one at github.com/settings/tokens → Fine-grained → only this repo → Contents: Read.) This is needed because the repo is private.
3. Run the cells top to bottom. Training takes roughly 30–60 minutes per model with `--sample 50000`.

Shared results sheet: https://docs.google.com/spreadsheets/d/1gUbY9_Q3I7ffzBZy49wnijjO6hGt10sTb3QMbCriWZQ/edit?usp=sharing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%env ARTIFACT_DIR=/content/drive/MyDrive/toxicity-artifacts
# ^ change this if the shared folder lives under 'Shared drives/...'
!mkdir -p $ARTIFACT_DIR

In [ ]:
# Clone (or update) the private repo using the GITHUB_TOKEN secret. The token is never printed or left in git config.
import os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/bwwarner/nlp-native-toxicity.git'
repo = '/content/nlp-native-toxicity'
if not os.path.exists(repo):
    subprocess.run(['git', 'clone', '-q', url, repo], check=True)
    subprocess.run(['git', '-C', repo, 'remote', 'set-url', 'origin', 'https://github.com/bwwarner/nlp-native-toxicity.git'], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '-q', url, 'main'], check=True)
%cd /content/nlp-native-toxicity
!git checkout -q main
!git log --oneline -1
!pip -q install -r requirements.txt

Make the train/val/test splits and the baseline **only if they aren't on Drive yet** (skipped automatically if they are).

In [ ]:
!test -f $ARTIFACT_DIR/splits/test.parquet && echo "splits already exist" || python -m src.data
!test -f $ARTIFACT_DIR/models/baseline.joblib && test -f $ARTIFACT_DIR/preds/baseline_test.csv && echo "baseline already exists" || python -m src.baseline
!ls $ARTIFACT_DIR/splits $ARTIFACT_DIR/models

In [ ]:
!python -m src.templates
!python -m src.augment

Train both DistilBERT models. **Use the same `--sample` for both** so augmentation is the only difference.

In [ ]:
!python -m src.train_distilbert --sample 50000

In [ ]:
!python -m src.train_distilbert --aug --sample 50000

In [ ]:
!python -m src.audit
!python -m src.demo_cases

In [ ]:
from IPython.display import Image, display
display(Image('results/fig_neutral_by_term.png'))
display(Image('results/fig_rates_by_group.png'))

**Save results to Drive** (Colab deletes everything else when it shuts down). Then download the `results` folder from Drive into the repo's `results/` folder on your laptop.

In [ ]:
!mkdir -p $ARTIFACT_DIR/results && cp -r results/* $ARTIFACT_DIR/results/ && ls $ARTIFACT_DIR/results

Optional: upload the DistilBERT weights to Hugging Face for the demo. Paste a **write** token from huggingface.co/settings/tokens when asked (don't save it in the notebook).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m src.push_to_hub --user YOUR_HF_USERNAME